In [1]:
import pandas as pd
import numpy as np
import os
import torch
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split


input_file = r'D:\disertatie\CSE_CIC_IDS_2018\Wednesday.csv'
output_path = r'D:\disertatie\preprocesare_anomalii_2018'
os.makedirs(output_path, exist_ok=True)

print("---  PROCESARE CSE-CIC-IDS-2018  ---")


df = pd.read_csv(input_file)

if 'Timestamp' in df.columns:
    df = df.drop(columns=['Timestamp'])

print(" Eliminare NaN și Infinity (văzute în fișier)...")
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)


df['Label_Bin'] = df['Label'].apply(lambda x: 0 if x == 'Benign' else 1)

print(f" Clase detectate: {df['Label'].unique()}")
print(f" Distribuție: Normal: {sum(df['Label_Bin']==0)} | Atacuri: {sum(df['Label_Bin']==1)}")


X = df.drop(columns=['Label', 'Label_Bin']).apply(pd.to_numeric, errors='coerce')
X.dropna(axis=1, inplace=True) 

X_normal = X[df['Label_Bin'] == 0]

X_train_normal, X_test_normal = train_test_split(X_normal, test_size=0.2, random_state=42)

X_anomalies = X[df['Label_Bin'] == 1]
X_test_final = pd.concat([X_test_normal, X_anomalies])

y_train_final = np.zeros(len(X_train_normal)) 
y_test_final = np.concatenate([np.zeros(len(X_test_normal)), np.ones(len(X_anomalies))])

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_normal)
X_test_scaled = scaler.transform(X_test_final)
print(f" Date partajate: Train_Normal={len(X_train_scaled)}, Test_Total={len(X_test_scaled)} (Normal={len(X_test_normal)}, Atac={len(X_anomalies)})")

print(" Salvare fișiere .pt și scaler...")
torch.save(torch.tensor(X_train_scaled, dtype=torch.float32), os.path.join(output_path, 'X_train_2018.pt'))
torch.save(torch.tensor(y_train_final, dtype=torch.float32), os.path.join(output_path, 'y_train_2018.pt'))
torch.save(torch.tensor(X_test_scaled, dtype=torch.float32), os.path.join(output_path, 'X_test_2018.pt'))
torch.save(torch.tensor(y_test_final, dtype=torch.float32), os.path.join(output_path, 'y_test_2018.pt'))

joblib.dump(scaler, os.path.join(output_path, 'scaler_2018.pkl'))

print(f"Gata! Datele sunt salvate în: {output_path}")

---  PROCESARE CSE-CIC-IDS-2018  ---
 Eliminare NaN și Infinity (văzute în fișier)...
 Clase detectate: <StringArray>
['Benign', 'FTP-BruteForce', 'SSH-Bruteforce']
Length: 3, dtype: str
 Distribuție: Normal: 663808 | Atacuri: 380943
 Date partajate: Train_Normal=531046, Test_Total=513705 (Normal=132762, Atac=380943)
 Salvare fișiere .pt și scaler...
Gata! Datele sunt salvate în: D:\disertatie\preprocesare_anomalii_2018
